# Scraping for the practice
These codes in this file aim to conduct web scraping in a Budget Bill, fiscal year 2023-24.

In [2]:
pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests
from bs4 import BeautifulSoup
import re
import time
import csv

In [4]:
def scrape_ca_bill_practice(url):
    # 1. Check the URL
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Error: Could not retrieve the page (Status code: {response.status_code})")
        return

    # 2. Parse the HTML
    soup = BeautifulSoup(response.text, 'html.parser')

    # 3. Extract the bill title and content
    # Take the bill title from the first <h1> tag, if it exists
    bill_title = soup.find('h1').text.strip() if soup.find('h1') else "Title not found"
    
    # Take the bill content from the div with id 'bill_all', if it exists
    bill_content = soup.find('div', id='bill_all')
    
    print(f"【Bill Title】: {bill_title}\n")
    if bill_content:
        # Display the first 500 characters of the content
        print("【Bill Content Excerpt】:") 
        print(bill_content.text.strip()[:500] + "...")
    else:
        print("Content not found. The ID may have changed.")

In [5]:
# Execute
target_url = "https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id=202320240SB101"
scrape_ca_bill_practice(target_url)

【Bill Title】: Bill Text

【Bill Content Excerpt】:
Senate Bill
                           							
                           No. 101



                        CHAPTER 12

An act making appropriations for the support of the government of the State of California and for several public purposes
                  in accordance with the provisions of Section 12 of Article IV of the Constitution of the State of California, relating to
                  the state budget, to take effect immediately, budget bill.
               									
               ...


In [6]:
# Extract the content of the bill and the amount of budget allocated to the bill
# サンプルのHTML構造を想定（実際には requests で取得した soup を使用）

def extract_budget_item(soup):
    # 1. 条件に合致する td を探す
    # colspan="7" かつ width="326" の td を指定
    target_td = soup.find('td', attrs={'colspan': '7', 'width': '326'})

    if not target_td:
        print("対象の td が見つかりませんでした。")
        return None

    # 2. その中の div を取得
    target_div = target_td.find('div')
    if target_div:
        full_text = target_div.get_text(strip=True)
        
        # 3. テキストを分割する
        # 「—」(全角ダッシュ/エムダッシュ) または 「--」(ハイフン2つ) で分割されることが多いです
        # どちらにも対応できるように正規表現(re)を使うと安全です
        import re
        parts = re.split(r'[—－-]{1,}', full_text) # ダッシュやハイフンで分割
        
        if len(parts) >= 2:
            item_number = parts[0].strip()      # 0120-011-0001
            item_description = parts[1].strip() # For support of Assembly
            
            return {
                "number": item_number,
                "description": item_description
            }
    
    return None

# 実行例
result = extract_budget_item(soup)
if result:
    print(f"番号: {result['number']}")
    print(f"内容: {result['description']}")

NameError: name 'soup' is not defined

In [ ]:
url = "https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id=202320240SB101"
headers = {'User-Agent': 'Mozilla/5.0'}

# 1. ページを取得
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 2. 条件に合う td をすべて取得
all_tds = soup.find_all('td', attrs={'colspan': '7', 'width': '326'})

# 3. 最初の5つだけに絞ってループを回す
for i, td in enumerate(all_tds[:5]):
    div = td.find('div')
    if div:
        full_text = div.get_text(strip=True)
        
        # 修正ポイント: maxsplit=1 を指定して、最初の1箇所だけで分割する
        # また、予算案でよく使われる「長いダッシュ(—)」を優先的に狙います
        if "—" in full_text:
            parts = full_text.split("—", 1)
        else:
            # 長いダッシュがない場合は、正規表現で「2つ以上続くハイフン」などで試みる
            parts = re.split(r'\s{2,}|--', full_text, maxsplit=1)
        
        if len(parts) >= 2:
            num = parts[0].strip()
            desc = parts[1].strip()
            print(f"Item {i+1}: {num} | {desc}")
        else:
            # 分割できなかった場合、そのまま表示して確認
            print(f"Item {i+1} (分割失敗): {full_text}")

Item 1: 0110-001-0001 | For support of Senate
                              ........................
Item 2: 0120-011-0001 | For support of Assembly
                              ........................
Item 3: 0130-021-0001 | For support of Legislative Analyst’s Office
                              ........................
Item 4: 0160-001-0001 | For support of Legislative Counsel Bureau
                              ........................
Item 5: 0160-001-9740 | For support of Legislative Counsel Bureau, payable from the Central Service Cost Recovery Fund
                              ........................


In [ ]:
def get_budget_items(bill_id, limit=None):
    """
    指定された法案IDから予算項目を抽出し、辞書のリストで返す関数
    :param bill_id: 法案ID (例: '202320240SB101')
    :param limit: 取得する最大件数 (Noneの場合は全件)
    :return: 抽出データのリスト
    """
    url = f"https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id={bill_id}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # エラーがあればここで停止
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 条件に合う td をすべて取得
        target_tds = soup.find_all('td', attrs={'colspan': '7', 'width': '326'})
        
        # リミットが設定されている場合はスライス
        if limit:
            target_tds = target_tds[:limit]
            
        results = []
        
        for td in target_tds:
            div = td.find('div')
            if div:
                full_text = div.get_text(strip=True)
                
                # 「—」で分割、回数は1回のみ
                if "—" in full_text:
                    parts = full_text.split("—", 1)
                else:
                    # ダッシュがない場合のフォールバック（2つ以上の空白やハイフン）
                    parts = re.split(r'\s{2,}|--', full_text, maxsplit=1)
                
                if len(parts) >= 2:
                    results.append({
                        "item_number": parts[0].strip(),
                        "description": parts[1].strip()
                    })
        
        return results

    except Exception as e:
        print(f"エラーが発生しました: {e}")
        return []

# --- 使い方 ---
# 最初の10件を取得してみる
bill_data = get_budget_items("202320240SB101", limit=10)

for entry in bill_data:
    print(f"[{entry['item_number']}] {entry['description']}")

[0110-001-0001] For support of Senate
                              ........................
[0120-011-0001] For support of Assembly
                              ........................
[0130-021-0001] For support of Legislative Analyst’s Office
                              ........................
[0160-001-0001] For support of Legislative Counsel Bureau
                              ........................
[0160-001-9740] For support of Legislative Counsel Bureau, payable from the Central Service Cost Recovery Fund
                              ........................
[0250-001-0001] For support of Judicial Branch
                              ........................
[0250-001-0044] For support of Judicial Branch, payable from the Motor Vehicle Account, State Transportation Fund
                              ........................
[0250-001-0159] For support of Judicial Branch, payable from the State Trial Court Improvement and Modernization Fund
                             

In [ ]:
def get_budget_items_with_amounts(bill_id, limit=None):
    url = f"https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id={bill_id}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 1. まずは「説明」が入っている td をすべて見つける
    description_tds = soup.find_all('td', attrs={'colspan': '7', 'width': '326'})
    
    if limit:
        description_tds = description_tds[:limit]
        
    results = []
    
    for td in description_tds:
        # --- A. 番号と説明の抽出 ---
        div = td.find('div')
        if not div: continue
        
        full_text = div.get_text(strip=True)
        if "—" in full_text:
            parts = full_text.split("—", 1)
        else:
            parts = re.split(r'\s{2,}|--', full_text, maxsplit=1)
            
        if len(parts) < 2: continue
        
        item_num = parts[0].strip()
        description = parts[1].strip()
        
        # --- B. 金額の抽出 ---
        # この td の親要素(tr)に移動し、そこから金額用の td (width="80") を探す
        parent_tr = td.find_parent('tr')
        amount = "0"
        if parent_tr:
            # align="right" かつ width="80" の td を探す
            amount_td = parent_tr.find('td', attrs={'align': 'right', 'width': '80'})
            if amount_td:
                amount = amount_td.get_text(strip=True)
        
        results.append({
            "item_number": item_num,
            "description": description,
            "amount": amount
        })
        
    return results

# 実行
data = get_budget_items_with_amounts("202320240SB101", limit=5)
for item in data:
    print(f"ID: {item['item_number']}")
    print(f"Explanation: {item['description']}")
    print(f"Amount: {item['amount']}")
    print("-" * 30)

ID: 0110-001-0001
Explanation: For support of Senate
                              ........................
Amount: 177,325,000
------------------------------
ID: 0120-011-0001
Explanation: For support of Assembly
                              ........................
Amount: 233,648,000
------------------------------
ID: 0130-021-0001
Explanation: For support of Legislative Analyst’s Office
                              ........................
Amount: 0
------------------------------
ID: 0160-001-0001
Explanation: For support of Legislative Counsel Bureau
                              ........................
Amount: 172,870,000
------------------------------
ID: 0160-001-9740
Explanation: For support of Legislative Counsel Bureau, payable from the Central Service Cost Recovery Fund
                              ........................
Amount: 35,805,000
------------------------------


In [ ]:
# Output as a csv file
def save_to_csv(data, filename):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=["item_number", "description", "amount"])
        writer.writeheader()
        
        for item in data:
            # description から 2つ以上続くドット「..」を削除
            # \. はドットそのものを指し、{2,} は「2回以上の繰り返し」を意味します
            clean_description = re.sub(r'\.{2,}', '', item['description']).strip()
            
            # 念のため、残ってしまった末尾のドットや空白を掃除
            clean_description = clean_description.rstrip('. ')
            
            # データを書き込む（元の item 自体を書き換えないよう、新しい辞書を作るのが安全）
            writer.writerow({
                "item_number": item['item_number'],
                "description": clean_description,
                "amount": item['amount']
            })

# 実行
save_to_csv(data, "budget_items.csv")

In [ ]:
# 1. サブカテゴリ1 (width=214) とその金額 (width=74)
def extract_sub1(td_element):
    # テキスト抽出
    div = td_element.find('div')
    full_text = div.get_text(strip=True) if div else ""
    # 「0960-Support」を分割
    match = re.match(r'^(\d{4})[-\s]+(.*)', full_text)
    code = match.group(1) if match else ""
    name = match.group(2) if match else full_text
    
    # 金額抽出 (同じ行の width=74)
    parent_tr = td_element.find_parent('tr')
    amount = "0"
    if parent_tr:
        amount_td = parent_tr.find('td', attrs={'width': '74', 'align': 'right'})
        if amount_td:
            amount = amount_td.get_text(strip=True)
            
    return {"code": code, "name": name, "amount": amount}

# 2. サブカテゴリ2 (width=124) とその金額 (width=66)
def extract_sub2(td_element):
    # テキスト抽出
    div = td_element.find('div')
    full_text = div.get_text(strip=True) if div else ""
    # 「101001-Salaries」を分割 (6桁の数字を想定)
    match = re.match(r'^(\d{6})[-\s]+(.*)', full_text)
    code = match.group(1) if match else ""
    name = match.group(2) if match else full_text
    
    # 金額抽出 (同じ行の width=66)
    parent_tr = td_element.find_parent('tr')
    amount = "0"
    if parent_tr:
        amount_td = parent_tr.find('td', attrs={'width': '66', 'align': 'right'})
        if amount_td:
            amount = amount_td.get_text(strip=True)
            
    return {"code": code, "name": name, "amount": amount}

In [7]:
def scrape_budget_hierarchy(bill_id):
    url = f"https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id={bill_id}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # すべての行を取得
    all_rows = soup.find_all('tr')
    
    results = []
    # 現在どの親項目にいるかを保持する変数
    current_main = {"id": "", "name": ""}
    current_sub1 = {"id": "", "name": ""}

    for tr in all_rows:
        # 1. メイン項目 (width=326)
        main_td = tr.find('td', attrs={'width': '326', 'colspan': '7'})
        if main_td:
            div = main_td.find('div')
            if div:
                text = div.get_text(strip=True)
                parts = text.split("—", 1) if "—" in text else [text, ""]
                current_main = {"id": parts[0].strip(), "name": re.sub(r'\.{2,}', '', parts[1]).strip(". ")}
                
                amount_td = tr.find('td', attrs={'width': '80', 'align': 'right'})
                amount = amount_td.get_text(strip=True) if amount_td else "0"
                
                results.append({
                    "Level": "Main", "Parent_ID": "", 
                    "Item_ID": current_main["id"], "Name": current_main["name"], "Amount": amount
                })
                continue

        # 2. サブカテゴリ1 (width=214)
        sub1_td = tr.find('td', attrs={'width': '214', 'colspan': '4'})
        if sub1_td:
            div = sub1_td.find('div')
            if div:
                text = div.get_text(strip=True)
                match = re.match(r'^(\d{4})[-\s]+(.*)', text)
                current_sub1 = {
                    "id": match.group(1) if match else "",
                    "name": re.sub(r'\.{2,}', '', match.group(2) if match else text).strip(". ")
                }
                
                amount_td = tr.find('td', attrs={'width': '74', 'align': 'right'})
                amount = amount_td.get_text(strip=True) if amount_td else "0"
                
                results.append({
                    "Level": "Sub1", "Parent_ID": current_main["id"], 
                    "Item_ID": current_sub1["id"], "Name": current_sub1["name"], "Amount": amount
                })
                continue

        # 3. サブカテゴリ2 (width=124)
        sub2_td = tr.find('td', attrs={'width': '124', 'colspan': '2'})
        if sub2_td:
            div = sub2_td.find('div')
            if div:
                text = div.get_text(strip=True)
                match = re.match(r'^(\d{6})[-\s]+(.*)', text)
                s2_id = match.group(1) if match else ""
                s2_name = re.sub(r'\.{2,}', '', match.group(2) if match else text).strip(". ")
                
                amount_td = tr.find('td', attrs={'width': '66', 'align': 'right'})
                amount = amount_td.get_text(strip=True) if amount_td else "0"
                
                results.append({
                    "Level": "Sub2", "Parent_ID": current_sub1["id"], 
                    "Item_ID": s2_id, "Name": s2_name, "Amount": amount
                })

    return results

# 保存用関数
def save_hierarchical_csv(data, filename):
    keys = ["Level", "Parent_ID", "Item_ID", "Name", "Amount"]
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(data)

# 実行
data = scrape_budget_hierarchy("202320240SB101")
save_hierarchical_csv(data, "ca_budget_full.csv")
print(f"{len(data)} 件のデータを保存しました。")

4533 件のデータを保存しました。


In [8]:
# Save as CSV file 
def save_to_budget_csv(data, filename="ca_budget_list.csv"):
    """
    抽出した階層データをCSVファイルとして保存する。
    """
    # CSVのヘッダー（列名）を定義
    fieldnames = ["Level", "Parent_ID", "Item_ID", "Name", "Amount"]
    
    try:
        with open(filename, mode='w', newline='', encoding='utf-8-sig') as file:
            # utf-8-sig を使うと、Excelで開いた時の文字化けを防げます
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            
            # ヘッダーを書き込む
            writer.writeheader()
            
            # データを1行ずつ書き込む
            for row in data:
                writer.writerow(row)
                
        print(f"成功: '{filename}' に一覧表を保存しました。({len(data)}件)")
        
    except Exception as e:
        print(f"CSV保存中にエラーが発生しました: {e}")

# --- 実行手順 ---

# 1. データの取得（前回の関数を使用）
# data = scrape_budget_hierarchy("202320240SB101")

# 2. CSVに出力
save_to_budget_csv(data)

成功: 'ca_budget_list.csv' に一覧表を保存しました。(4533件)
